# 01 – Daten finden und laden (NGC 3132, NIRCam)

**Was:** Die Stufe-3-Bilder (`_i2d.fits`) der sechs NIRCam-Filter aus Programm 2733 in MAST finden, ihre Größe prüfen und herunterladen.

**Warum:** Eine JWST-Beobachtung hat oft tausende Datenprodukte: Rohdaten, Zwischenstufen, Kataloge. Wir brauchen pro Filter genau eine Datei, das fertig kombinierte Bild. Vergleich: Aus einem ganzen Mikroskop-Ordner (Einzelkacheln, Z-Stapel, Metadaten) nimmt man nur das gestitchte Endbild.

**Ablauf:**
1. Beobachtungen suchen (wie im Setup-Check, jetzt nur NIRCam)
2. Produktliste holen und auf `_i2d.fits` filtern
3. Größe prüfen, **bevor** geladen wird
4. Zuerst die kleinste Datei laden (Testlauf), dann den Rest

**Ergebnis:** 6 i2d-Dateien (aus 1956 Produkten herausgefiltert), alle `PUBLIC`, zusammen ≈2,49 GB, gespeichert in `data/ngc3132_nircam/` (nicht im Git). Alle Downloads mit Status `COMPLETE`.

In [1]:
from pathlib import Path
from astroquery.mast import Observations

# Data live outside git (see .gitignore). Notebook runs in notebooks/, so go one level up.
DATA_DIR = Path("..") / "data" / "ngc3132_nircam"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Download folder:", DATA_DIR.resolve())

Download folder: /Users/inu/Projects/astro-pipeline/data/ngc3132_nircam


## 1. Beobachtungen suchen

`query_criteria` durchsucht die **Beobachtungsliste**, also *was* gemessen wurde, noch keine Dateien.
- `calib_level=3`: nur die fertig kombinierten Produkte, nicht die Einzelbelichtungen
- `instrument_name="NIRCAM/IMAGE"`: den Wert haben wir im Setup-Check gesehen

Erwartung: 6 Zeilen, eine pro Filter.

In [2]:
obs = Observations.query_criteria(
    obs_collection="JWST",
    proposal_id="2733",
    instrument_name="NIRCAM/IMAGE",
    calib_level=3,
)
print(len(obs), "NIRCam observations")
obs["obs_id", "filters", "t_exptime"]

6 NIRCam observations


obs_id,filters,t_exptime
str36,str11,float64
jw02733-o001_t001_nircam_clear-f187n,F187N,2319.144
jw02733-o001_t001_nircam_clear-f090w,F090W,1460.2000000000003
jw02733-o001_t001_nircam_clear-f356w,F356W,1460.2000000000003
jw02733-o001_t001_nircam_clear-f212n,F212N,2319.144
jw02733-o001_t001_nircam_f405n-f444w,F444W;F405N,2319.144
jw02733-o001_t001_nircam_f444w-f470n,F444W;F470N,2319.144


## 2. Produktliste holen und filtern

`get_product_list` liefert **alle Dateien**, die zu diesen Beobachtungen gehören, auch die Einzelbelichtungen, aus denen das Endbild gebaut wurde. Deshalb ist die Liste lang.

Mit `filter_products` behalten wir nur:
- `productType="SCIENCE"`: Bilddaten, keine Hilfsdateien
- `productSubGroupDescription="I2D"`: das kombinierte, auf ein Himmelsraster gebrachte Bild (*i*maging, *2D*)
- `calib_level=[3]`: nur Stufe 3

Die Spalte `dataRights` zeigt, ob die Daten öffentlich sind (`PUBLIC`). Bei einer Early Release Observation sollten sie es sein.

In [3]:
products = Observations.get_product_list(obs)
print(len(products), "products in total")

i2d = Observations.filter_products(
    products,
    productType="SCIENCE",
    productSubGroupDescription="I2D",
    calib_level=[3],
)
print(len(i2d), "i2d products after filtering")
i2d["productFilename", "size", "dataRights"]

1956 products in total
6 i2d products after filtering


productFilename,size,dataRights
str51,int64,str6
jw02733-o001_t001_nircam_clear-f090w_i2d.fits,673997760,PUBLIC
jw02733-o001_t001_nircam_clear-f356w_i2d.fits,155134080,PUBLIC
jw02733-o001_t001_nircam_f405n-f444w_i2d.fits,155134080,PUBLIC
jw02733-o001_t001_nircam_f444w-f470n_i2d.fits,155134080,PUBLIC
jw02733-o001_t001_nircam_clear-f187n_i2d.fits,674118720,PUBLIC
jw02733-o001_t001_nircam_clear-f212n_i2d.fits,674259840,PUBLIC


## 3. Größe prüfen

`size` steht in Bytes. Wir rechnen in MB/GB um und sortieren aufsteigend. Wenn hier deutlich mehr als 6 Dateien oder eine unerwartet große Summe auftauchen, **nicht** weiterlaufen lassen. Dann schauen wir zuerst, was in der Liste steckt.

In [4]:
i2d.sort("size")
for row in i2d:
    print(f"{row['productFilename']:<55} {row['size']/1e6:8.1f} MB")
print(f"\nTotal: {i2d['size'].sum()/1e9:.2f} GB")

jw02733-o001_t001_nircam_clear-f356w_i2d.fits              155.1 MB
jw02733-o001_t001_nircam_f405n-f444w_i2d.fits              155.1 MB
jw02733-o001_t001_nircam_f444w-f470n_i2d.fits              155.1 MB
jw02733-o001_t001_nircam_clear-f090w_i2d.fits              674.0 MB
jw02733-o001_t001_nircam_clear-f187n_i2d.fits              674.1 MB
jw02733-o001_t001_nircam_clear-f212n_i2d.fits              674.3 MB

Total: 2.49 GB


## 4a. Testlauf: kleinste Datei laden

Erst eine Datei, um zu prüfen, ob der Download funktioniert und wo die Datei landet. Das ist wie ein Testbild, bevor man die Zeitreihe startet.

`flat=True` legt die Datei direkt in `DATA_DIR`, ohne die verschachtelte MAST-Ordnerstruktur. Das Manifest zeigt pro Datei `Status` (sollte `COMPLETE` sein) und den lokalen Pfad.

In [5]:
manifest = Observations.download_products(i2d[:1], download_dir=str(DATA_DIR), flat=True)
manifest

Local Path,Status,Message,URL
str68,str8,object,object
../data/ngc3132_nircam/jw02733-o001_t001_nircam_clear-f356w_i2d.fits,COMPLETE,None,None


## 4b. Restliche Dateien laden

Nur ausführen, wenn 4a geklappt hat. Den Schalter dafür auf `True` setzen. Bereits vorhandene Dateien überspringt astroquery normalerweise. Ich bin mir aber nicht ganz sicher, ob das bei `flat=True` in jeder Version greift. Beim Ausführen sieht man es an der Meldung.

In [6]:
DOWNLOAD_ALL = True   # set to True after the test download worked

if DOWNLOAD_ALL:
    manifest_all = Observations.download_products(i2d, download_dir=str(DATA_DIR), flat=True)
    display(manifest_all["Local Path", "Status"])
else:
    print("Skipped. Set DOWNLOAD_ALL = True to download all i2d files.")

INFO: Found cached file ../data/ngc3132_nircam/jw02733-o001_t001_nircam_clear-f356w_i2d.fits with expected size 155139840. [astroquery.query]


Local Path,Status
str68,str8
../data/ngc3132_nircam/jw02733-o001_t001_nircam_clear-f356w_i2d.fits,COMPLETE
../data/ngc3132_nircam/jw02733-o001_t001_nircam_f405n-f444w_i2d.fits,COMPLETE
../data/ngc3132_nircam/jw02733-o001_t001_nircam_f444w-f470n_i2d.fits,COMPLETE
../data/ngc3132_nircam/jw02733-o001_t001_nircam_clear-f090w_i2d.fits,COMPLETE
../data/ngc3132_nircam/jw02733-o001_t001_nircam_clear-f187n_i2d.fits,COMPLETE
../data/ngc3132_nircam/jw02733-o001_t001_nircam_clear-f212n_i2d.fits,COMPLETE


## 5. Kontrolle: was liegt jetzt lokal?

In [8]:
for f in sorted(DATA_DIR.glob("*_i2d.fits")):
    print(f"{f.name:<55} {f.stat().st_size/1e6:8.1f} MB")

jw02733-o001_t001_nircam_clear-f090w_i2d.fits              674.0 MB
jw02733-o001_t001_nircam_clear-f187n_i2d.fits              674.1 MB
jw02733-o001_t001_nircam_clear-f212n_i2d.fits              674.1 MB
jw02733-o001_t001_nircam_clear-f356w_i2d.fits              155.1 MB
jw02733-o001_t001_nircam_f405n-f444w_i2d.fits              155.1 MB
jw02733-o001_t001_nircam_f444w-f470n_i2d.fits              155.1 MB


## Fazit

- Die Filterkette `SCIENCE` + `I2D` + `calib_level=[3]` reduziert 1956 Produkte zuverlässig auf genau eine Datei pro Filter.
- Kurzwellen-Bilder (F090W, F187N, F212N) sind ≈674 MB groß, Langwellen-Bilder (F356W, F405N, F470N) ≈155 MB. Das Verhältnis ≈4,3 passt zu den Pixelgrößen (0,031″ vs. 0,063″): halbe Pixelkante → ≈4× so viele Pixel. Wichtig für die Wahl des Referenzrasters in Etappe 3.
- Kleine Auffälligkeit: Die tatsächlichen Dateigrößen weichen um einige kB von der Spalte `size` der Produktliste ab (z. B. F212N: 674 121 600 statt 674 259 840 Byte). astroquery selbst erwartet beim Cache-Check genau die lokale Größe, und alle Downloads melden `COMPLETE` – vermutlich ist nur die Katalog-Angabe nicht ganz aktuell (z. B. nach einer Neuprozessierung). Nicht sicher; beim Öffnen in Etappe 2 sehen wir, ob die Dateien intakt sind.
- Der Download überspringt bereits vorhandene Dateien („Found cached file“), auch mit `flat=True`.

**Nächster Schritt:** Etappe 2 – eine i2d-Datei öffnen und verstehen (Extensions, Header, WCS, Histogramm).